In [1]:
import pandas as pd
import os
import gzip
import numpy as np
import logging
from tqdm import tqdm
from functools import partial
from multiprocessing import Pool, cpu_count

# Configure logging to provide informative output
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')

In [2]:
def read_profile(filepath):
    """
    Reads a gzipped profile file. Uses pandas' built-in compression handling.
    """
    if not isinstance(filepath, str) or not os.path.exists(filepath):
        logging.info(f"Profile file not found or path is invalid: {filepath}")
        return None

    profile_df = pd.read_csv(filepath, sep="\t", compression="gzip", dtype={'contig': str, 'position': 'int64'})
    # Ensure all required columns are present for analysis
    required_cols = ['contig', 'position', 'A', 'C', 'G', 'T']
    if not all(col in profile_df.columns for col in required_cols):
        logging.warning(f"File {filepath} is missing required columns. Skipping.")
        return None
    return profile_df

def get_major_alleles(profile_df):
    """
    Vectorized function to find major alleles in a profile dataframe.
    This is much faster than iterating row-by-row with .apply().
    Returns a pandas Series where each value is a list of major alleles.
    """
    allele_cols = ['A', 'C', 'G', 'T']
    
    # Extract counts array, filling NaNs for robustness
    counts = profile_df[allele_cols].fillna(0).to_numpy(dtype=int)
    
    # Determine max count per row
    max_counts = counts.max(axis=1)
    
    # Identify rows with any coverage
    has_coverage = max_counts > 0
    
    # Build boolean mask where arr == max_count for each row, using the
    # clear [:, None] syntax for broadcasting (from user suggestion).
    # We only consider rows that have coverage.
    mask = (counts == max_counts[:, None]) & (has_coverage[:, None])
    
    # Construct list of alleles per row using the boolean mask
    alleles = np.array(allele_cols)
    major_alleles_list = [alleles[row_mask].tolist() for row_mask in mask]
    
    return pd.Series(major_alleles_list, index=profile_df.index)

def check_allele_match(row):
    """
    Helper function to check for a match between lists of alleles.
    Returns True if there is any overlap (non-empty intersection).
    """
    # A site must be present in the isolate to be considered for a match.
    if not row['present']:
        return False
        
    act_alleles = row['major_act']
    iso_alleles = row['major_iso']
    
    # Ensure both are non-empty lists before checking for intersection.
    if not act_alleles or not iso_alleles:
        return False
        
    # Return True if there's any overlap between the two lists of major alleles.
    return len(set(act_alleles).intersection(set(iso_alleles))) > 0


def compare_profiles(row, significant_p_sites=None, significant_q_sites=None):
    """
    Main comparison logic for a single row, designed to be called in parallel.
    Now includes separate checks for p-value and q-value significant sites.
    """
    if significant_p_sites is None:
        significant_p_sites = set()
    if significant_q_sites is None:
        significant_q_sites = set()

    isolate_path = row["profile_path_isolate"]
    actual_path = row["profile_path_actual"]

    actual_df = read_profile(actual_path)
    isolate_df = read_profile(isolate_path)
    
    assert not actual_df.duplicated(["contig", "position"]).any(), "actual_df has duplicated sites"
    assert not isolate_df.duplicated(["contig", "position"]).any(), "isolate_df has duplicated sites"

    note = ""
    if actual_df is None or actual_df.empty:
        note = "Actual file not found or empty"
    elif isolate_df is None or isolate_df.empty:
        note = "Isolate file not found or empty"

    if note:
        return (note, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan), None

    actual_df["major_act"] = get_major_alleles(actual_df)
    isolate_df["major_iso"] = get_major_alleles(isolate_df)

    merged = pd.merge(
        actual_df[["contig", "position", "major_act"]],
        isolate_df[["contig", "position", "major_iso"]],
        on=["contig", "position"],
        how="left",
    )

    # Use the explicit len(x) > 0 check for presence.
    merged['present'] = merged['major_iso'].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )
    merged["match"] = merged.apply(check_allele_match, axis=1)

    # --- New Significance Analysis ---
    merged_sites = pd.MultiIndex.from_frame(merged[['contig', 'position']])
    
    merged['is_significant_by_p_value'] = merged_sites.isin(significant_p_sites)
    merged['is_present_and_significant_by_p_value'] = merged['present'] & merged['is_significant_by_p_value']
    
    merged['is_significant_by_q_value'] = merged_sites.isin(significant_q_sites)
    merged['is_present_and_significant_by_q_value'] = merged['present'] & merged['is_significant_by_q_value']

    
    n_positions_actual = len(merged)
    n_positions_present = merged["present"].sum()
    n_major_matches = merged["match"].sum()
    n_significant_p = merged['is_significant_by_p_value'].sum()
    n_present_significant_p = merged['is_present_and_significant_by_p_value'].sum()
    n_significant_q = merged['is_significant_by_q_value'].sum()
    n_present_significant_q = merged['is_present_and_significant_by_q_value'].sum()
    
    # Calculate matches specifically within the significant sites that are present in the isolate
    n_major_matches_p_sites = (merged['match'] & merged['is_present_and_significant_by_p_value']).sum()
    n_major_matches_q_sites = (merged['match'] & merged['is_present_and_significant_by_q_value']).sum()


    summary_stats = (
        "Success",
        n_positions_actual,
        n_positions_present,
        n_major_matches,
        n_significant_p,
        n_present_significant_p,
        n_significant_q,
        n_present_significant_q,
        n_major_matches_p_sites,
        n_major_matches_q_sites
    )

    return summary_stats, merged

In [3]:
# --- Step A: Load and Prepare Metadata ---
logging.info("Loading and merging metadata files...")

isolate_fPath = "/scratch/gpfs/AMOELLER/Phocaeicola_AlleleFlux_Metadata.txt"
isolate_df = pd.read_csv(isolate_fPath, sep="\t")
isolate_df["time"] = isolate_df["time"].replace({"Pre-Treatment": "pre", "End": "end"})
isolate_df["group"] = isolate_df["group"].replace({"Control": "control", "Fat": "fat"})
isolate_df.drop(["bam_path"], axis=1, inplace=True)

actual_fPath = "/scratch/gpfs/sg4230/popgentoolkit/metadata_md_bam.tsv"
actual_df = pd.read_csv(actual_fPath, sep="\t")
actual_df.drop(["cage", "bam_path", "diet", "day", "Sex"], axis=1, inplace=True)

# Merge the dataframes
df = isolate_df.merge(
    actual_df,
    on=["group", "replicate", "time", "subjectID"],
    suffixes=("_isolate", "_actual"),
)


2025-07-31 11:18:18 - INFO - Loading and merging metadata files...


In [4]:
# --- Step B: Load and Filter Significance Data ---
mag_id = "SLG221_DASTool_bins_41"
logging.info("Loading and filtering p-value summary file...")
p_value_summary_path = "/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_final_copy_dnds/longitudinal/p_value_summary/pre_end-fat_control/p_value_summary_two_sample_paired_pre_end.tsv"
df_p_value = pd.read_csv(p_value_summary_path, sep="\t", dtype={'mag_id': str, 'contig': str, 'position': 'int64'})

# Filter the significance dataframe for the specific MAG ID
df_p_value_filtered = df_p_value[df_p_value['mag_id'] == mag_id]
logging.info(f"Filtered p-value summary to {len(df_p_value_filtered):,} rows for mag_id {mag_id}.")


# Filter for sites significant by p-value
significant_p_df = df_p_value_filtered[
    (df_p_value_filtered['min_p_value'] < 0.05) &
    (df_p_value_filtered['test_type'] == 'two_sample_paired_tTest')
]
significant_p_sites_set = set(zip(significant_p_df['contig'], significant_p_df['position']))
logging.info(f"Found {len(significant_p_sites_set):,} sites significant by p-value (p < 0.05).")

# Filter for sites significant by q-value
significant_q_df = df_p_value_filtered[
    (df_p_value_filtered['q_value'] < 0.05) &
    (df_p_value_filtered['test_type'] == 'two_sample_paired_tTest')
]
significant_q_sites_set = set(zip(significant_q_df['contig'], significant_q_df['position']))
logging.info(f"Found {len(significant_q_sites_set):,} sites significant by q-value (q < 0.05).")


2025-07-31 11:18:19 - INFO - Loading and filtering p-value summary file...
2025-07-31 11:18:22 - INFO - Filtered p-value summary to 23,926 rows for mag_id SLG221_DASTool_bins_41.
2025-07-31 11:18:22 - INFO - Found 2,371 sites significant by p-value (p < 0.05).
2025-07-31 11:18:22 - INFO - Found 301 sites significant by q-value (q < 0.05).


In [5]:
# --- Step C: Construct File Paths ---
logging.info("Constructing profile file paths...")
base_dir_isolate = "/scratch/gpfs/AMOELLER/diet_manip/isolates/single_timepoint/profiles"
base_dir_actual = "/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_final_copy_dnds/longitudinal/profiles"

df["profile_path_isolate"] = df["sample_id_isolate"].apply(
    lambda sample_id: f"{base_dir_isolate}/{sample_id}/{sample_id}_{mag_id}_profiled.tsv.gz" if pd.notna(sample_id) else None
)
df["profile_path_actual"] = df["sample_id_actual"].apply(
    lambda sample_id: f"{base_dir_actual}/{sample_id}/{sample_id}_{mag_id}_profiled.tsv.gz" if pd.notna(sample_id) else None
)
df

2025-07-31 11:18:22 - INFO - Constructing profile file paths...


,sample_id_isolate,subjectID,group,replicate,time,sample_id_actual,profile_path_isolate,profile_path_actual
0,1102_A4_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
1,1102_E6_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
2,1104_D1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
3,1104_E1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
4,1104_E6_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
5,1104_F1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
6,421_C9_10471829_HT5CLAFX5.fna,542,control,4,pre,SLG421,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
7,SLG1205_F4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
8,SLG1205_G4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...
9,SLG1207_B7_10473203_HT53MAFX5.fna,564,fat,8,end,SLG1207,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...


In [6]:
# --- Step D: Run Parallel Comparison ---
output_dir_merged = "/scratch/gpfs/AMOELLER/sidd/isolate_analysis"
os.makedirs(output_dir_merged, exist_ok=True)
logging.info(f"Running comparison... Detailed merged files will be saved to '{output_dir_merged}/'")

rows_to_process = [row for _, row in df.iterrows()]
num_processes = 20
logging.info(f"Starting parallel comparison for {len(df)} pairs using {num_processes} processes...")

# Use functools.partial to "bake in" the sets of significant sites for each worker process
task_function = partial(compare_profiles, significant_p_sites=significant_p_sites_set, significant_q_sites=significant_q_sites_set)

with Pool(processes=num_processes) as pool:
    parallel_results = list(
        tqdm(
            pool.imap(task_function, rows_to_process),
            total=len(rows_to_process),
            desc="Comparing Profiles",
        )
    )

logging.info("Parallel comparison finished.")


2025-07-31 11:18:26 - INFO - Running comparison... Detailed merged files will be saved to '/scratch/gpfs/AMOELLER/sidd/isolate_analysis/'
2025-07-31 11:18:26 - INFO - Starting parallel comparison for 31 pairs using 20 processes...
Comparing Profiles: 100%|██████████| 31/31 [01:59<00:00,  3.85s/it]
2025-07-31 11:20:26 - INFO - Parallel comparison finished.


In [7]:
summary_list = [res[0] for res in parallel_results]
merged_df_list = [res[1] for res in parallel_results]

In [9]:
output_dir_merged = os.path.join("/scratch/gpfs/AMOELLER/sidd/isolate_analysis", mag_id)

# Create subdirectories for the new output files to keep things organized
output_dir_all = os.path.join(output_dir_merged, f"{mag_id}_all_merged_sites")
output_dir_p_sig = os.path.join(output_dir_merged, f"{mag_id}_p_significant_sites")
output_dir_q_sig = os.path.join(output_dir_merged, f"{mag_id}_q_significant_sites")
os.makedirs(output_dir_all, exist_ok=True)
os.makedirs(output_dir_p_sig, exist_ok=True)
os.makedirs(output_dir_q_sig, exist_ok=True)

logging.info("Saving detailed merged dataframes...")
for i, merged_df in enumerate(tqdm(merged_df_list, desc="Saving Files")):
    if merged_df is not None:
        row = df.iloc[i]
        act_id = row["sample_id_actual"]
        iso_id = row["sample_id_isolate"]
        base_filename = f"merged_{act_id}_vs_{iso_id}"

        # Save the full merged file to its own subdirectory as a TSV
        output_path = os.path.join(output_dir_all, f"{base_filename}.tsv")
        merged_df.to_csv(output_path, index=False, sep='\t')

        # Save the p-value significant subset as a TSV
        p_sig_df = merged_df[merged_df['is_significant_by_p_value']]
        if not p_sig_df.empty:
            p_sig_output_path = os.path.join(output_dir_p_sig, f"{base_filename}_p_significant.tsv")
            p_sig_df.to_csv(p_sig_output_path, index=False, sep='\t')

        # Save the q-value significant subset as a TSV
        q_sig_df = merged_df[merged_df['is_significant_by_q_value']]
        if not q_sig_df.empty:
            q_sig_output_path = os.path.join(output_dir_q_sig, f"{base_filename}_q_significant.tsv")
            q_sig_df.to_csv(q_sig_output_path, index=False, sep='\t')

logging.info("All merged files saved.")

2025-07-31 11:21:40 - INFO - Saving detailed merged dataframes...
Saving Files: 100%|██████████| 31/31 [04:31<00:00,  8.77s/it]
2025-07-31 11:26:12 - INFO - All merged files saved.


In [10]:
results_df = pd.DataFrame(
    summary_list,
    columns=[
        "comparison_status",
        "n_positions_actual",
        "n_positions_present",
        "n_major_matches",
        "n_significant_p_in_actual",
        "n_significant_p_and_present_in_isolate",
        "n_significant_q_in_actual",
        "n_significant_q_and_present_in_isolate",
        "n_major_matches_p_sites",
        "n_major_matches_q_sites"
    ],
)
final_df = df.join(results_df)

In [11]:
# Use .div() and .fillna(0) for safe division to prevent NaN/inf results.
final_df["site_overlap_fraction"] = final_df["n_positions_present"].div(final_df["n_positions_actual"]).fillna(0)
final_df["major_allele_match_fraction"] = final_df["n_major_matches"].div(final_df["n_positions_present"]).fillna(0)
final_df["p_value_site_retention_fraction"] = final_df["n_significant_p_and_present_in_isolate"].div(final_df["n_significant_p_in_actual"]).fillna(0)
final_df["q_value_site_retention_fraction"] = final_df["n_significant_q_and_present_in_isolate"].div(final_df["n_significant_q_in_actual"]).fillna(0)
final_df["major_allele_match_fraction_p_sites"] = final_df["n_major_matches_p_sites"].div(final_df["n_significant_p_and_present_in_isolate"]).fillna(0)
final_df["major_allele_match_fraction_q_sites"] = final_df["n_major_matches_q_sites"].div(final_df["n_significant_q_and_present_in_isolate"]).fillna(0)

final_df

,sample_id_isolate,subjectID,group,replicate,time,sample_id_actual,profile_path_isolate,profile_path_actual,comparison_status,n_positions_actual,...,n_significant_q_in_actual,n_significant_q_and_present_in_isolate,n_major_matches_p_sites,n_major_matches_q_sites,site_overlap_fraction,major_allele_match_fraction,p_value_site_retention_fraction,q_value_site_retention_fraction,major_allele_match_fraction_p_sites,major_allele_match_fraction_q_sites
0,1102_A4_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,4921793,...,301,300,1676,205,0.999946,0.999640,0.997868,0.996678,0.716239,0.683333
1,1102_E6_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,4921793,...,301,300,1665,205,0.997961,0.999611,0.994456,0.996678,0.713979,0.683333
2,1104_D1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,5061830,...,301,257,1911,242,0.915144,0.999657,0.855335,0.853821,0.942308,0.941634
3,1104_E1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,5061830,...,301,294,2150,279,0.985875,0.999791,0.973851,0.976744,0.931139,0.948980
4,1104_E6_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,5061830,...,301,299,2203,279,0.997708,0.999820,0.995361,0.993355,0.933475,0.933110
5,1104_F1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,5061830,...,301,298,2168,282,0.992158,0.999804,0.984395,0.990033,0.928877,0.946309
6,421_C9_10471829_HT5CLAFX5.fna,542,control,4,pre,SLG421,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,5061518,...,301,300,2137,245,0.995771,0.999800,0.993674,0.996678,0.907046,0.816667
7,SLG1205_F4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,4833859,...,298,298,1405,152,0.999938,0.999535,0.997011,1.000000,0.601713,0.510067
8,SLG1205_G4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,4833859,...,298,298,1446,150,0.999647,0.999543,0.996157,1.000000,0.619803,0.503356
9,SLG1207_B7_10473203_HT53MAFX5.fna,564,fat,8,end,SLG1207,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_f...,Success,5055643,...,300,300,2030,281,0.999923,0.999756,0.997889,1.000000,0.859077,0.936667


In [12]:
# Define the columns for which to calculate statistics
metrics_to_agg = [
    "site_overlap_fraction",
    "major_allele_match_fraction",
    "p_value_site_retention_fraction",
    "q_value_site_retention_fraction",
    "major_allele_match_fraction_p_sites",
    "major_allele_match_fraction_q_sites"
]
grouped_summary = final_df.groupby(['time', 'group'])[metrics_to_agg].agg(['mean', 'std'])
grouped_summary

site_overlap_fraction           major_allele_match_fraction  \
                              mean       std                        mean   
time group                                                                 
end  control              0.999385  0.000832                    0.999655   
     fat                  0.985510  0.027114                    0.999768   
pre  control              0.999014  0.001824                    0.999635   
     fat                  0.993542  0.011654                    0.999772   

                       p_value_site_retention_fraction            \
                   std                            mean       std   
time group                                                         
end  control  0.000085                        0.996658  0.001462   
     fat      0.000058                        0.973378  0.045623   
pre  control  0.000092                        0.995944  0.002225   
     fat      0.000061                        0.986713  0.017355   

             q_value_site_retention_fraction            \
                                        mean       std   
time group                                               
end  control                        0.998754  0.001719   
     fat                            0.977063  0.044918   
pre  control                        0.997336  0.002788   
     fat                            0.991279  0.011500   

             major_allele_match_fraction_p_sites            \
                                            mean       std   
time group                                                   
end  control                            0.728137  0.080300   
     fat                                0.899676  0.033442   
pre  control                            0.708318  0.111184   
     fat                                0.860213  0.046700   

             major_allele_match_fraction_q_sites            
                                            mean       std  
time group                                                  
end  control                            0.652875  0.091098  
     fat                                0.943077  0.010997  
pre  control                            0.628500  0.105280  
     fat                                0.746765  0.064632

In [13]:
# Save the grouped summary to a new file
grouped_summary_output_path = os.path.join(output_dir_merged, f"{mag_id}_grouped_summary_statistics.tsv")
grouped_summary.to_csv(grouped_summary_output_path, sep='\t')
logging.info(f"Grouped summary statistics saved to {grouped_summary_output_path}")


2025-07-31 11:26:12 - INFO - Grouped summary statistics saved to /scratch/gpfs/AMOELLER/sidd/isolate_analysis/SLG221_DASTool_bins_41/SLG221_DASTool_bins_41_grouped_summary_statistics.tsv


In [14]:
summary_output_path = os.path.join(output_dir_merged, f"{mag_id}_summary_comparison_results.tsv")
final_df.to_csv(summary_output_path, index=False, sep="\t")

### Checking the results

In [32]:
for i, merged_df in enumerate(merged_df_list):
    row = df.iloc[i]
    # print(df)
    act_id = row["sample_id_actual"]
    iso_id = row["sample_id_isolate"]
    base_filename = f"merged_{act_id}_vs_{iso_id}"
    p_sig_df = merged_df[merged_df['is_significant_by_q_value']]
    if i==0:
        print(base_filename)
        break

p_sig_df

merged_SLG1102_vs_1102_A4_10471829_HT5CLAFX5.fna


,contig,position,major_act,major_iso,present,match,is_significant_by_p_value,is_present_and_significant_by_p_value,is_significant_by_q_value,is_present_and_significant_by_q_value
267924,SLG221_DASTool_bins_41.fa_k141_123387,14542,[G],[G],True,True,True,True,True,True
267990,SLG221_DASTool_bins_41.fa_k141_123387,14608,[T],[G],True,False,True,True,True,True
268047,SLG221_DASTool_bins_41.fa_k141_123387,14665,[A],[G],True,False,True,True,True,True
268056,SLG221_DASTool_bins_41.fa_k141_123387,14674,[G],[T],True,False,True,True,True,True
268059,SLG221_DASTool_bins_41.fa_k141_123387,14677,[G],[T],True,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...
4910292,SLG221_DASTool_bins_41.fa_k141_90926,42430,[G],[G],True,True,True,True,True,True
4910294,SLG221_DASTool_bins_41.fa_k141_90926,42432,[T],[T],True,True,True,True,True,True
4910320,SLG221_DASTool_bins_41.fa_k141_90926,42458,[C],[C],True,True,True,True,True,True
4910325,SLG221_DASTool_bins_41.fa_k141_90926,42463,[T],[T],True,True,True,True,True,True


In [26]:
df_1102_A4_isolate = pd.read_csv("/scratch/gpfs/AMOELLER/diet_manip/isolates/single_timepoint/profiles/1102_A4_10471829_HT5CLAFX5.fna/1102_A4_10471829_HT5CLAFX5.fna_SLG221_DASTool_bins_41_profiled.tsv.gz", sep="\t")

In [27]:
df_SLG1102 = pd.read_csv("/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_final/longitudinal/profiles/SLG1102/SLG1102_SLG221_DASTool_bins_41_profiled.tsv.gz", sep="\t")

In [28]:
contigs_of_interest = ["SLG221_DASTool_bins_41.fa_k141_123387"]
position_of_interest = [14542,14608, 14665, 14674, 14677]

In [29]:
df_SLG1102[(df_SLG1102['contig'].isin(contigs_of_interest)) & (df_SLG1102['position'].isin(position_of_interest))]

,contig,position,ref_base,total_coverage,A,C,G,T,N,gene_id
267924,SLG221_DASTool_bins_41.fa_k141_123387,14542,G,7,3,0,4,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
267990,SLG221_DASTool_bins_41.fa_k141_123387,14608,G,13,0,0,4,9,0,SLG221_DASTool_bins_41.fa_k141_123387_18
268047,SLG221_DASTool_bins_41.fa_k141_123387,14665,G,19,14,0,5,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
268056,SLG221_DASTool_bins_41.fa_k141_123387,14674,T,18,0,0,13,5,0,SLG221_DASTool_bins_41.fa_k141_123387_18
268059,SLG221_DASTool_bins_41.fa_k141_123387,14677,T,18,0,0,13,5,0,SLG221_DASTool_bins_41.fa_k141_123387_18


In [30]:
df_1102_A4_isolate[(df_1102_A4_isolate['contig'].isin(contigs_of_interest)) & (df_1102_A4_isolate['position'].isin(position_of_interest))]

,contig,position,ref_base,total_coverage,A,C,G,T,N,gene_id
272821,SLG221_DASTool_bins_41.fa_k141_123387,14542,G,29,0,0,29,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
272887,SLG221_DASTool_bins_41.fa_k141_123387,14608,G,22,0,0,22,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
272944,SLG221_DASTool_bins_41.fa_k141_123387,14665,G,15,0,0,15,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
272953,SLG221_DASTool_bins_41.fa_k141_123387,14674,T,16,0,0,0,16,0,SLG221_DASTool_bins_41.fa_k141_123387_18
272956,SLG221_DASTool_bins_41.fa_k141_123387,14677,T,17,0,0,0,17,0,SLG221_DASTool_bins_41.fa_k141_123387_18
